# EMG Channel Reduction — Feature-Based Clustering

Groups 24 sEMG channels into **8 clusters** based on the similarity of their extracted feature profiles.

**What this captures:** discriminative similarity — channels that produce the same feature
response across all gesture classes carry redundant information for the classifier.

**Key design choice:** uses the full 8-dimensional feature stream (MAV, RMS, WL, ZC, SSC, VAR, MNF, MDF)
z-scored per feature so all 8 dimensions contribute equally to the correlation.

**Pipeline:**
1. Load preprocessed .mat files, slide a 250-sample window (shift=50)
2. Extract 8 features x 24 channels per window -> X shape (N_windows, 192)
3. Reshape to (N_windows, 24, 8), z-score per feature, compute 24x24 correlation matrix
4. Agglomerative clustering — average linkage, correlation distance
5. Cut at k=8, select one representative per cluster via medoid

In [1]:
import sys, os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import squareform   # converts square dist matrix <-> condensed 1-D vector
from sklearn.metrics import silhouette_score    # validates whether k=8 is a good choice
from sklearn.preprocessing import StandardScaler  # z-scores features so all 8 contribute equally

# make the shared emg_loader module importable from the parent repo root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
PUBLIC    = os.path.join(REPO_ROOT, 'public')
if PUBLIC not in sys.path:
    sys.path.insert(0, PUBLIC)

from emg_loader import load_all_subjects, extract_features_from_subjects
print('Imports OK.')

Imports OK.


In [2]:
DATA_DIR          = '/Volumes/KRIS/data/UG_per_subject'   # folder of per-subject .mat files
N_CH_CLUSTERS     = 8     # target number of channel groups
N_CHANNELS        = 24    # total sEMG channels in the putEMG setup
N_FEATURES_PER_CH = 8     # MAV, RMS, WL, ZC, SSC, VAR, MNF, MDF
WINDOW_SIZE       = 250   # samples per window (~49 ms at 5120 Hz)
WINDOW_SHIFT      = 50    # step between windows (~10 ms), gives ~80% overlap
SAMPLING_RATE     = 5120.0
GESTURE_NAMES     = ['G1', 'G2', 'G3', 'G6', 'G7', 'G8', 'G9']
FEATURE_NAMES     = ['MAV', 'RMS', 'WL', 'ZC', 'SSC', 'VAR', 'MNF', 'MDF']

In [3]:
# load_all_subjects returns list of (name, X, Y) where X shape: (N_reps, 1, 24, 1500)
subjects = load_all_subjects(DATA_DIR)
print(f'Loaded {len(subjects)} subjects')

# extract_features_from_subjects slides a window over every rep,
# computes 8 features per channel per window, and returns a flat feature matrix
# X shape: (N_windows, 192)  where 192 = 24 channels * 8 features
# feature layout: [8 feats ch0 | 8 feats ch1 | ... | 8 feats ch23]
X, y = extract_features_from_subjects(
    subjects,
    window_size   = WINDOW_SIZE,
    window_shift  = WINDOW_SHIFT,
    sampling_rate = SAMPLING_RATE,
)
print(f'Feature matrix: X={X.shape}, y={y.shape}')

# reshape to (N_windows, 24, 8) — one 8-dim feature vector per channel per window
# this makes it easy to operate on channels independently
X_ch = X.reshape(-1, N_CHANNELS, N_FEATURES_PER_CH)

Loading 44 subject file(s) from: /Volumes/KRIS/data/UG_per_subject

  → emg_gestures_03_U.mat  (280 samples)
  → emg_gestures_04_U.mat  (280 samples)
  → emg_gestures_05_U.mat  (280 samples)
  → emg_gestures_06_U.mat  (280 samples)
  → emg_gestures_07_U.mat  (280 samples)
  → emg_gestures_08_U.mat  (280 samples)
  → emg_gestures_09_U.mat  (280 samples)
  → emg_gestures_10_U.mat  (280 samples)
  → emg_gestures_11_U.mat  (280 samples)
  → emg_gestures_12_U.mat  (280 samples)
  → emg_gestures_13_U.mat  (280 samples)
  → emg_gestures_14_U.mat  (280 samples)
  → emg_gestures_15_U.mat  (280 samples)
  → emg_gestures_16_U.mat  (280 samples)
  → emg_gestures_17_U.mat  (280 samples)
  → emg_gestures_18_U.mat  (280 samples)
  → emg_gestures_19_U.mat  (280 samples)
  → emg_gestures_20_U.mat  (280 samples)
  → emg_gestures_22_U.mat  (280 samples)
  → emg_gestures_23_U.mat  (280 samples)
  → emg_gestures_24_U.mat  (280 samples)
  → emg_gestures_25_U.mat  (280 samples)
  → emg_gestures_26_U.mat  (28

KeyboardInterrupt: 

In [ ]:
# --- Build the improved channel correlation matrix ---
#
# Naive approach: collapse 8 features to 1 scalar per window (e.g. mean),
# then correlate. This throws away 7/8 of the feature information.
#
# Better approach (used here): keep the full 8-dim feature stream per channel,
# z-score each feature independently so WL (large values) doesn't dominate
# MAV or ZC (small values), then correlate the full streams.
#
# (N, 24, 8) -> transpose -> (24, N, 8) -> reshape -> (24, N*8)
# each row is one channel's complete feature stream across all windows
channel_flat = X_ch.transpose(1, 0, 2).reshape(N_CHANNELS, -1)

# StandardScaler computes mean/std per column, so we transpose, scale, transpose back
# after scaling, every feature dimension has mean=0 and std=1 across windows
channel_flat_z = StandardScaler().fit_transform(channel_flat.T).T  # still (24, N*8)

# corrcoef expects (variables, observations) -> channels are variables, N*8 stream is observations
corr_matrix = np.corrcoef(channel_flat_z)   # (24, 24)
np.fill_diagonal(corr_matrix, 1.0)          # ensure exact 1s on diagonal

# quick sanity check
mask = ~np.eye(N_CHANNELS, dtype=bool)
print(f'Off-diagonal corr  min={corr_matrix[mask].min():.3f}  '
      f'max={corr_matrix[mask].max():.3f}  '
      f'mean={corr_matrix[mask].mean():.3f}')

ch_labels = [f'Ch{i+1}' for i in range(N_CHANNELS)]
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, vmin=-1, vmax=1, cmap='RdBu_r',
            xticklabels=ch_labels, yticklabels=ch_labels,
            ax=ax, linewidths=0.3, linecolor='grey')
ax.set_title('Channel-Channel Pearson Correlation  (Feature-Based, z-scored)', fontsize=13)
ax.tick_params(axis='x', rotation=90, labelsize=8)
ax.tick_params(axis='y', rotation=0,  labelsize=8)
plt.tight_layout()
plt.savefig('feat_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Class-conditional mean activation per channel.
# For each gesture class, compute the average feature magnitude across all windows.
# Channels with nearly identical rows across all 7 gestures are redundant —
# they don't add discriminative information beyond what another channel already provides.
#
# collapse the 8 features per channel to one scalar (mean) for visualisation purposes
X_ch_scalar = X_ch.mean(axis=2)   # (N_windows, 24)

class_means = np.zeros((7, N_CHANNELS))
for g in range(7):
    class_means[g] = X_ch_scalar[y == g].mean(axis=0)  # mean over all windows of gesture g

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(class_means, xticklabels=ch_labels, yticklabels=GESTURE_NAMES,
            cmap='viridis', ax=ax, linewidths=0.3)
ax.set_title('Mean Channel Activation per Gesture Class', fontsize=13)
ax.set_xlabel('Channel')
ax.set_ylabel('Gesture')
ax.tick_params(axis='x', rotation=90, labelsize=9)
plt.tight_layout()
plt.savefig('feat_class_conditional.png', dpi=150)
plt.show()

In [ ]:
def plot_dendrogram_with_cut(
    Z,
    k,
    n_channels=24,
    title='Channel Dendrogram',
    save_path=None,
    representative_channels=None,
):
    # -----------------------------------------------------------------------
    # Draw the agglomerative clustering dendrogram and overlay a dashed red
    # line at the height that produces exactly k clusters.
    #
    # How the cut height works:
    #   Z is a (n-1, 4) linkage matrix. Each row records one merge:
    #     [left_child, right_child, merge_distance, cluster_size]
    #   Rows are ordered from smallest to largest merge distance.
    #   Z[-(k-1), 2] is the distance at which we go from k+1 -> k clusters,
    #   so cutting at this height gives exactly k groups.
    # -----------------------------------------------------------------------
    cut_height = Z[-(k - 1), 2]   # height that yields exactly k clusters
    ch_labels  = [f'Ch{i+1}' for i in range(n_channels)]

    fig, ax = plt.subplots(figsize=(14, 5))

    # color_threshold colours each subtree below the cut differently
    # so you can visually see the k groups
    dendrogram(
        Z,
        labels=ch_labels,
        color_threshold=cut_height,
        ax=ax,
        leaf_rotation=0,
        leaf_font_size=10,
    )

    # dashed horizontal line showing exactly where the tree is cut
    ax.axhline(
        cut_height, color='red', linestyle='--', linewidth=1.5,
        label=f'k={k} cut  (dist={cut_height:.4f})',
    )
    ax.legend(fontsize=10)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Channel')
    ax.set_ylabel('Correlation distance  (1 - r)')

    # place a red star below each leaf that was selected as cluster representative
    if representative_channels:
        ylim   = ax.get_ylim()
        offset = (ylim[1] - ylim[0]) * 0.07   # push star slightly below the x-axis
        for tick in ax.get_xticklabels():
            ch_idx = int(tick.get_text().replace('Ch', '')) - 1
            if ch_idx in representative_channels:
                ax.text(
                    tick.get_position()[0],
                    ylim[0] - offset,
                    chr(9733),            # unicode filled star
                    ha='center', va='top', fontsize=13, color='red',
                    transform=ax.transData,
                )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Cut height : {cut_height:.4f}')
    print(f'Clusters   : {k}')

In [ ]:
# --- Step 1: convert correlation to a distance metric ---
# distance = 1 - r, so r=1 (identical channels) -> dist=0,
# and r=0 (uncorrelated) -> dist=1
# clip at 0 prevents tiny negative values from floating-point subtraction
dist_matrix = np.clip(1.0 - corr_matrix, 0.0, None)

# FIX: enforce exact symmetry before calling squareform
# np.corrcoef can produce corr[i,j] != corr[j,i] by ~1e-16 due to float arithmetic
# squareform checks D == D.T elementwise and raises ValueError if they differ at all
# averaging D and D.T collapses those micro-differences to zero
dist_matrix = (dist_matrix + dist_matrix.T) / 2

# diagonal must be exactly 0 (distance from a channel to itself)
np.fill_diagonal(dist_matrix, 0.0)

# --- Step 2: build the linkage matrix ---
# squareform converts the (24, 24) matrix to a condensed 1-D vector (upper triangle)
# this is the format scipy linkage() requires as input
condensed = squareform(dist_matrix)

# average linkage: distance between two clusters = mean of all pairwise member distances
# good middle ground between single linkage (chaining) and complete linkage (compactness)
Z = linkage(condensed, method='average')

# --- Step 3: cut the tree at k clusters ---
# fcluster finds the minimum cut height that produces exactly k non-empty clusters
# subtract 1 to convert from scipy's 1-indexed labels to 0-indexed
cluster_labels = fcluster(Z, N_CH_CLUSTERS, criterion='maxclust') - 1

print('Cluster composition:')
for c in range(N_CH_CLUSTERS):
    members = np.where(cluster_labels == c)[0]
    print(f'  Cluster {c}: channels {list(members + 1)}')

In [ ]:
# Representative selection: medoid strategy
# The medoid is the channel most central to its cluster —
# the one with the highest average Pearson r to every other member.
# More robust than a centroid because we stay in the original channel space
# (no interpolated 'average channel' that may not correspond to a real electrode).
representative_channels = []

for c in range(N_CH_CLUSTERS):
    members  = np.where(cluster_labels == c)[0]          # 0-indexed channel indices
    sub_corr = corr_matrix[np.ix_(members, members)]     # sub-matrix for this cluster only
    avg_corr = sub_corr.mean(axis=1)                     # mean corr of each member to all others
    rep      = members[np.argmax(avg_corr)]              # pick the most central channel
    representative_channels.append(rep)
    print(f'  Cluster {c}: {list(members + 1)}  ->  Ch{rep + 1} (idx {rep})')

# sort so channel indices are in ascending order
representative_channels = sorted(representative_channels)
print()
print(f'Final representatives (0-indexed): {representative_channels}')
print(f'Final representatives (1-indexed): {[c + 1 for c in representative_channels]}')

In [ ]:
# call the display function — change k here to explore other cut points
plot_dendrogram_with_cut(
    Z,
    k=N_CH_CLUSTERS,
    title=f'Feature-Based Dendrogram  —  average linkage  (k={N_CH_CLUSTERS})',
    save_path='feat_dendrogram.png',
    representative_channels=representative_channels,
)

In [ ]:
# Silhouette score measures how well each channel fits its assigned cluster
# vs. the nearest other cluster.  Range: -1 (wrong cluster) to +1 (well separated).
# Sweeping k=2..12 shows whether k=8 sits near a local maximum,
# which confirms it is a natural breakpoint in the channel similarity structure.
k_range    = range(2, 13)
sil_scores = []

for k in k_range:
    labels_k = fcluster(Z, k, criterion='maxclust') - 1
    # metric='precomputed' tells sklearn to treat dist_matrix as distances directly
    score    = silhouette_score(dist_matrix, labels_k, metric='precomputed')
    sil_scores.append(score)
    print(f'  k={k:>2}  silhouette={score:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range), sil_scores, marker='o', color='steelblue', linewidth=1.5)
ax.axvline(N_CH_CLUSTERS, color='red', linestyle='--', label=f'k={N_CH_CLUSTERS}')
ax.set_xlabel('Number of clusters  (k)')
ax.set_ylabel('Silhouette score')
ax.set_title('Silhouette Validation  —  Feature-Based')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('feat_silhouette.png', dpi=150)
plt.show()

In [ ]:
# persist results so other notebooks can load them without re-running clustering
np.save('feat_representative_channels.npy', np.array(representative_channels))  # shape (8,)
np.save('feat_cluster_labels.npy', cluster_labels)                               # shape (24,)
print('Saved:  feat_representative_channels.npy   feat_cluster_labels.npy')
print(f'Selected channels (1-indexed): {[c + 1 for c in representative_channels]}')